# YOLO26 訓練補強

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/16-training/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.6.1'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""Verified progressive-weight schedule in a small two-head regression example."""
import json
import torch
from torch import nn
from torch.nn import functional as F


def weights(epoch, epochs, final_many=.1):
    many = max(1 - epoch / max(epochs - 1, 1), 0) * (.8 - final_many) + final_many
    return many, 1 - many


def train(progressive, epochs=30, final_many=.1, fixed_weights=(.8, .2)):
    torch.manual_seed(7)  # paired initialization and fixed data
    features = torch.tensor([[-1.], [0.], [1.], [2.]])
    target = 2 * features + 1
    many, one = nn.Linear(1, 1), nn.Linear(1, 1)
    optimizer = torch.optim.SGD(list(many.parameters()) + list(one.parameters()), lr=.05)
    history = []
    first_step = {}
    for epoch in range(epochs):
        a, b = weights(epoch, epochs, final_many) if progressive else fixed_weights
        optimizer.zero_grad()
        lm = F.mse_loss(many(features), target)
        one_prediction = one(features)
        lo = F.mse_loss(one_prediction, target)
        if epoch == 0:
            first_step.update(weight=float(one.weight.detach()), bias=float(one.bias.detach()),
                              prediction=one_prediction.detach().flatten().tolist(), one_mse=float(lo.detach()), one_gain=b)
        loss = a * lm + b * lo
        loss.backward()
        assert many.weight.grad is not None and one.weight.grad is not None
        if epoch == 0:
            first_step.update(weighted_dw=float(one.weight.grad), weighted_dbias=float(one.bias.grad))
        optimizer.step()
        if epoch == 0:
            first_step.update(updated_weight=float(one.weight.detach()), updated_bias=float(one.bias.detach()))
        history.append((a, b, float(lo.detach())))
    return F.mse_loss(one(features), target).item(), history, first_step


def total_one_weight(history):
    """Sum of the one-head loss weight b over all epochs of one run."""
    return sum(b for _, b, _ in history)


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    final_many = .1  # Change only this value to .3 for the schedule exercise.
    fixed, fixed_history, _ = train(False)
    progressive, history, first_step = train(True, final_many=final_many)
    assert abs(history[0][0] - .8) < 1e-7 and abs(history[-1][0] - final_many) < 1e-7
    # The heads are independent: a scales only the many head's gradient, b only the one head's.
    # Matched control: fixed weights with the schedule's sum of b. Against fixed .8/.2 the schedule
    # differs in both shape and sum of b; against this control only the shape differs.
    matched_one = total_one_weight(history) / len(history)
    matched_weights = (1 - matched_one, matched_one)
    matched, matched_history, _ = train(False, fixed_weights=matched_weights)
    assert abs(total_one_weight(matched_history) - total_one_weight(history)) < 1e-9
    actual = torch.tensor([first_step[k] for k in ('weight', 'bias', 'one_mse', 'weighted_dw', 'weighted_dbias', 'updated_weight', 'updated_bias')])
    expected = torch.tensor([.318423, .313781, 5.866377, -1.146190, -.610803, .375733, .344321])
    assert torch.allclose(actual, expected, atol=2e-6, rtol=1e-6)
    print('many/one weight first:', tuple(round(v, 3) for v in history[0][:2]))
    print('many/one weight last:', tuple(round(v, 3) for v in history[-1][:2]))
    runs = ((f'fixed many/one {tuple(round(v, 3) for v in fixed_history[0][:2])}', fixed, fixed_history),
            ('progressive', progressive, history),
            (f'matched fixed many/one {tuple(round(v, 3) for v in matched_weights)}', matched, matched_history))
    for name, mse, run_history in runs:
        print(f'{name}: sum of b={total_one_weight(run_history):.3f}, one-head MSE={mse:.6f}')
    print('one-head first forward/backward/step:', json.dumps(first_step))
    # STAL-style eligibility only: retain the actual GT for regression.
    gt = torch.tensor([7., 7., 9., 9.])
    points = torch.tensor([[4., 4.], [12., 4.], [4., 12.], [12., 12.]])
    center = (gt[:2] + gt[2:]) / 2
    expanded_size = (gt[2:] - gt[:2]).clamp(min=16)
    pool_box = torch.cat((center - expanded_size / 2, center + expanded_size / 2))
    inside = lambda box: ((points > box[:2]) & (points < box[2:])).all(-1)
    assert inside(gt).sum() == 0 and inside(pool_box).sum() == 4
    assert gt.tolist() == [7., 7., 9., 9.]
    print('small-object eligible points original / expanded:', int(inside(gt).sum()), int(inside(pool_box).sum()))
    print('GT regression box remains:', gt.tolist())
    print('not a full YOLO26/STAL/MuSGD training reproduction')


if __name__ == '__main__':
    main()


many/one weight first: (0.8, 0.2)
many/one weight last: (0.1, 0.9)
fixed many/one (0.8, 0.2): sum of b=6.000, one-head MSE=0.663073
progressive: sum of b=16.500, one-head MSE=0.016285
matched fixed many/one (0.45, 0.55): sum of b=16.500, one-head MSE=0.016882
one-head first forward/backward/step: {"weight": 0.31842339038848877, "bias": 0.3137805461883545, "prediction": [-0.004642844200134277, 0.3137805461883545, 0.6322039365768433, 0.950627326965332], "one_mse": 5.866377353668213, "one_gain": 0.19999999999999996, "weighted_dw": -1.1461899280548096, "weighted_dbias": -0.6108031272888184, "updated_weight": 0.3757328987121582, "updated_bias": 0.34432071447372437}
small-object eligible points original / expanded: 0 4
GT regression box remains: [7.0, 7.0, 9.0, 9.0]
not a full YOLO26/STAL/MuSGD training reproduction
